## Task2: Restaurant Recommendation

In [3]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')

In [5]:
df = pd.read_csv("Dataset .csv")
df.head(10)

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229
5,18189371,Din Tai Fung,162,Mandaluyong City,"Ground Floor, Mega Fashion Hall, SM Megamall, ...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056314,14.583764,Chinese,...,Botswana Pula(P),No,No,No,No,3,4.4,Green,Very Good,336
6,6300781,Buffet 101,162,Pasay City,"Building K, SM By The Bay, Sunset Boulevard, M...","SM by the Bay, Mall of Asia Complex, Pasay City","SM by the Bay, Mall of Asia Complex, Pasay Cit...",120.979667,14.531333,"Asian, European",...,Botswana Pula(P),Yes,No,No,No,4,4.0,Green,Very Good,520
7,6301290,Vikings,162,Pasay City,"Building B, By The Bay, Seaside Boulevard, Mal...","SM by the Bay, Mall of Asia Complex, Pasay City","SM by the Bay, Mall of Asia Complex, Pasay Cit...",120.979333,14.540000,"Seafood, Filipino, Asian, European",...,Botswana Pula(P),Yes,No,No,No,4,4.2,Green,Very Good,677
8,6300010,Spiral - Sofitel Philippine Plaza Manila,162,Pasay City,"Plaza Level, Sofitel Philippine Plaza Manila, ...","Sofitel Philippine Plaza Manila, Pasay City","Sofitel Philippine Plaza Manila, Pasay City, P...",120.980090,14.552990,"European, Asian, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.9,Dark Green,Excellent,621
9,6314987,Locavore,162,Pasig City,"Brixton Technology Center, 10 Brixton Street, ...",Kapitolyo,"Kapitolyo, Pasig City",121.056532,14.572041,Filipino,...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,532


In [6]:
df['Cuisines'].fillna(
    df['Cuisines'].mode()[0],
    inplace=True
)

In [7]:
recommend_df = df[[
    'Restaurant Name',
    'City',
    'Cuisines',
    'Price range',
    'Aggregate rating'
]]

In [8]:
recommend_df.head()

,Restaurant Name,City,Cuisines,Price range,Aggregate rating
0,Le Petit Souffle,Makati City,"French, Japanese, Desserts",3,4.8
1,Izakaya Kikufuji,Makati City,Japanese,3,4.5
2,Heat - Edsa Shangri-La,Mandaluyong City,"Seafood, Asian, Filipino, Indian",4,4.4
3,Ooma,Mandaluyong City,"Japanese, Sushi",4,4.9
4,Sambo Kojin,Mandaluyong City,"Japanese, Korean",4,4.8


In [9]:
recommend_df['features'] = (
    recommend_df['Cuisines'].astype(str) + " " +
    recommend_df['City'].astype(str) + " " +
    recommend_df['Price range'].astype(str)
)

In [10]:
cv = CountVectorizer(stop_words='english')

feature_matrix = cv.fit_transform(
    recommend_df['features']
)

In [11]:
similarity = cosine_similarity(
    feature_matrix
)

In [12]:
def recommend_restaurant(name, top_n=5):

    if name not in recommend_df['Restaurant Name'].values:
        return "Restaurant not found"

    index = recommend_df[
        recommend_df['Restaurant Name'] == name
    ].index[0]

    scores = list(
        enumerate(similarity[index])
    )

    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )

    recommendations = []

    for i in scores[1:top_n+1]:

        recommendations.append({
            'Restaurant':
            recommend_df.iloc[i[0]]['Restaurant Name'],

            'Cuisine':
            recommend_df.iloc[i[0]]['Cuisines'],

            'City':
            recommend_df.iloc[i[0]]['City'],

            'Rating':
            recommend_df.iloc[i[0]]['Aggregate rating']
        })

    return pd.DataFrame(recommendations)

In [15]:
recommend_df['Restaurant Name'].sample(10)

8992                           Green Chick Chop
7487              Ye Old Bakery - The Claridges
8942                           Bats On Delivery
3682                            Mad Over Donuts
5308                               Tripti Foods
5115                          Lhasa Thali House
8173                                     Ni Hao
6524    Savannah Bar - Radisson Blu Plaza Delhi
5575                               Chaap Chaska
7278                            Da Pizza Palace
Name: Restaurant Name, dtype: object

In [16]:
recommend_restaurant("Barbeque Nation")

,Restaurant,Cuisine,City,Rating
0,Rasoi Ghar,Indian,Dubai,4.3
1,Carnival By Tresind,Indian,Dubai,4.9
2,Tresind - Nassima Royal Hotel,Indian,Dubai,4.9
3,SpiceKlub,"Indian, North Indian, Street Food",Dubai,4.4
4,Grub Shack,"Goan, Chinese, Indian, North Indian",Dubai,4.3


In [17]:
def recommend_restaurant(name, top_n=5):

    index = recommend_df[
        recommend_df['Restaurant Name'] == name
    ].index[0]

    scores = list(
        enumerate(similarity[index])
    )

    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )

    restaurant_indices = [
        i[0]
        for i in scores[1:50]
    ]

    recommendations = recommend_df.iloc[
        restaurant_indices
    ]

    recommendations = recommendations.sort_values(
        by='Aggregate rating',
        ascending=False
    )

    return recommendations.head(top_n)

In [18]:
recommend_restaurant("Barbeque Nation")

,Restaurant Name,City,Cuisines,Price range,Aggregate rating,features
590,Carnival By Tresind,Dubai,Indian,4,4.9,Indian Dubai 4
580,Punjab Grill,Abu Dhabi,"Indian, North Indian",4,4.9,"Indian, North Indian Abu Dhabi 4"
597,Tresind - Nassima Royal Hotel,Dubai,Indian,4,4.9,Indian Dubai 4
1839,Prankster,Gurgaon,"Modern Indian, North Indian",3,4.8,"Modern Indian, North Indian Gurgaon 3"
619,Rajasthan Al Malaki,Sharjah,"Indian, North Indian",3,4.8,"Indian, North Indian Sharjah 3"
